Create Spark Dataset

In [0]:
from pyspark.sql import functions as F

# Generate the original 1 million sales records
sales_df = (
    spark.range(1, 1000001)
    .select(
        F.col("id").alias("sales_id"),
        (F.col("id") % 10000 + 1).alias("customer_id"),

        F.when(F.col("id") % 5 == 0, "Electronics")
         .when(F.col("id") % 5 == 1, "Clothing")
         .when(F.col("id") % 5 == 2, "Furniture")
         .when(F.col("id") % 5 == 3, "Groceries")
         .otherwise("Sports").alias("category"),

        F.when(F.col("id") % 4 == 0, "Bengaluru")
         .when(F.col("id") % 4 == 1, "Hyderabad")
         .when(F.col("id") % 4 == 2, "Chennai")
         .otherwise("Mumbai").alias("region"),

        ((F.col("id") % 1000) + 100)
        .cast("double").alias("sales_amount"),

        F.when(F.col("id") % 10 == 0, "Cancelled")
         .otherwise("Completed").alias("status")
    )
)

print("Sales dataset created successfully!")
print("Total records:", sales_df.count())

Sales dataset created successfully!
Total records: 1000000


Introduce Data Quality Issues

In [0]:
# Create records containing intentional data quality issues
bad_data = [
    (1000001, 101, "Electronics", "Bengaluru", 500.0, "Completed"),
    (1000002, None, "Clothing", "Hyderabad", 300.0, "Completed"),
    (1000003, 102, "InvalidCategory", "Chennai", 250.0, "Completed"),
    (1000004, 103, "Furniture", "InvalidRegion", 400.0, "Completed"),
    (1000005, 104, "Sports", "Mumbai", -100.0, "Completed"),
    (1000006, 105, "Groceries", "Bengaluru", 200.0, "InvalidStatus"),
    (1000007, 106, "Sports", "Mumbai", None, "Completed"),
    (1000007, 106, "Sports", "Mumbai", 200.0, "Completed")
]

columns = [
    "sales_id",
    "customer_id",
    "category",
    "region",
    "sales_amount",
    "status"
]

bad_df = spark.createDataFrame(bad_data, columns)

# Combine original and intentionally incorrect records
quality_df = sales_df.unionByName(bad_df)

print("Data quality testing dataset created!")
print("Original records:", sales_df.count())
print("Records after adding test data:", quality_df.count())

display(bad_df)

Data quality testing dataset created!
Original records: 1000000
Records after adding test data: 1000008


sales_id,customer_id,category,region,sales_amount,status
1000001,101,Electronics,Bengaluru,500.0,Completed
1000002,null,Clothing,Hyderabad,300.0,Completed
1000003,102,InvalidCategory,Chennai,250.0,Completed
1000004,103,Furniture,InvalidRegion,400.0,Completed
1000005,104,Sports,Mumbai,-100.0,Completed
1000006,105,Groceries,Bengaluru,200.0,InvalidStatus
1000007,106,Sports,Mumbai,null,Completed
1000007,106,Sports,Mumbai,200.0,Completed


Check missing values

In [0]:
from pyspark.sql import functions as F

# Count missing values in each column
missing_values = quality_df.select([
    F.count(
        F.when(F.col(column).isNull(), column)
    ).alias(column)
    for column in quality_df.columns
])

print("Missing value report:")
display(missing_values)

Missing value report:


sales_id,customer_id,category,region,sales_amount,status
0,1,0,0,1,0


Check Duplicate Records

In [0]:
# Find duplicate sales IDs
duplicate_ids = (
    quality_df
    .groupBy("sales_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Number of duplicate sales IDs:", duplicate_ids.count())

print("Duplicate records:")
display(duplicate_ids)

Number of duplicate sales IDs: 1
Duplicate records:


sales_id,count
1000007,2


Check Missing Values

In [0]:
# Check missing values in every column

columns_to_check = [
    "sales_id",
    "customer_id",
    "category",
    "region",
    "sales_amount",
    "status"
]

missing_value_counts = quality_df.select([
    F.count(
        F.when(F.col(column).isNull(), column)
    ).alias(column)
    for column in columns_to_check
])

print("Missing value report:")
display(missing_value_counts)

Missing value report:


sales_id,customer_id,category,region,sales_amount,status
0,1,0,0,1,0


Validate Sales Amounts

In [0]:
# Find invalid sales amounts
invalid_amounts = quality_df.filter(
    (F.col("sales_amount") <= 0) |
    F.col("sales_amount").isNull()
)

print("Invalid sales amount records:", invalid_amounts.count())

display(invalid_amounts)

Invalid sales amount records: 2


sales_id,customer_id,category,region,sales_amount,status
1000005,104,Sports,Mumbai,-100.0,Completed
1000007,106,Sports,Mumbai,null,Completed


Validate Allowed Values

In [0]:
# Define allowed values
valid_categories = [
    "Electronics", "Clothing", "Furniture", "Groceries", "Sports"
]

valid_regions = [
    "Bengaluru", "Hyderabad", "Chennai", "Mumbai"
]

valid_statuses = [
    "Completed", "Cancelled"
]

# Find invalid categories
invalid_categories = quality_df.filter(
    ~F.col("category").isin(valid_categories)
)

# Find invalid regions
invalid_regions = quality_df.filter(
    ~F.col("region").isin(valid_regions)
)

# Find invalid statuses
invalid_statuses = quality_df.filter(
    ~F.col("status").isin(valid_statuses)
)

print("Invalid categories:", invalid_categories.count())
print("Invalid regions:", invalid_regions.count())
print("Invalid statuses:", invalid_statuses.count())

display(invalid_categories)
display(invalid_regions)
display(invalid_statuses)

Invalid categories: 1
Invalid regions: 1
Invalid statuses: 1


sales_id,customer_id,category,region,sales_amount,status
1000003,102,InvalidCategory,Chennai,250.0,Completed


sales_id,customer_id,category,region,sales_amount,status
1000004,103,Furniture,InvalidRegion,400.0,Completed


sales_id,customer_id,category,region,sales_amount,status
1000006,105,Groceries,Bengaluru,200.0,InvalidStatus


Validate Customer IDs

In [0]:
# Find invalid customer IDs
invalid_customers = quality_df.filter(
    F.col("customer_id").isNull() |
    (F.col("customer_id") <= 0)
)

print("Invalid customer ID records:", invalid_customers.count())

display(invalid_customers)

Invalid customer ID records: 1


sales_id,customer_id,category,region,sales_amount,status
1000002,null,Clothing,Hyderabad,300.0,Completed


Create a Data Quality Validation Report

In [0]:
# Collect validation results

validation_results = [
    ("Missing Customer IDs", invalid_customers.count()),
    ("Missing Sales Amounts", quality_df.filter(F.col("sales_amount").isNull()).count()),
    ("Duplicate Sales IDs", duplicate_ids.count()),
    ("Invalid Sales Amounts", invalid_amounts.count()),
    ("Invalid Categories", invalid_categories.count()),
    ("Invalid Regions", invalid_regions.count()),
    ("Invalid Statuses", invalid_statuses.count())
]

# Create validation report
validation_report = spark.createDataFrame(
    validation_results,
    ["Validation Rule", "Failed Records"]
)

# Add PASS / FAIL status
validation_report = validation_report.withColumn(
    "Result",
    F.when(F.col("Failed Records") == 0, "PASS")
     .otherwise("FAIL")
)

print("DATA QUALITY VALIDATION REPORT")

display(validation_report)

DATA QUALITY VALIDATION REPORT


Validation Rule,Failed Records,Result
Missing Customer IDs,1,FAIL
Missing Sales Amounts,1,FAIL
Duplicate Sales IDs,1,FAIL
Invalid Sales Amounts,2,FAIL
Invalid Categories,1,FAIL
Invalid Regions,1,FAIL
Invalid Statuses,1,FAIL


Identify All Invalid Records

In [0]:
# Define all data quality failure conditions

invalid_condition = (
    F.col("sales_id").isNull() |
    F.col("customer_id").isNull() |
    (F.col("customer_id") <= 0) |
    F.col("category").isNull() |
    (~F.col("category").isin(valid_categories)) |
    F.col("region").isNull() |
    (~F.col("region").isin(valid_regions)) |
    F.col("sales_amount").isNull() |
    (F.col("sales_amount") <= 0) |
    F.col("status").isNull() |
    (~F.col("status").isin(valid_statuses)) |
    F.col("sales_id").isin(
        [row["sales_id"] for row in duplicate_ids.select("sales_id").collect()]
    )
)

# Extract invalid records
invalid_records = quality_df.filter(invalid_condition)

print("Total invalid records:", invalid_records.count())

display(invalid_records.orderBy("sales_id"))

Total invalid records: 7


sales_id,customer_id,category,region,sales_amount,status
1000002,null,Clothing,Hyderabad,300.0,Completed
1000003,102,InvalidCategory,Chennai,250.0,Completed
1000004,103,Furniture,InvalidRegion,400.0,Completed
1000005,104,Sports,Mumbai,-100.0,Completed
1000006,105,Groceries,Bengaluru,200.0,InvalidStatus
1000007,106,Sports,Mumbai,null,Completed
1000007,106,Sports,Mumbai,200.0,Completed


Create a Cleaned Dataset

In [0]:
# Remove invalid records
cleaned_sales = quality_df.filter(~invalid_condition)

print("Original records:", quality_df.count())
print("Invalid records:", invalid_records.count())
print("Cleaned records:", cleaned_sales.count())

print("Cleaned dataset:")
display(cleaned_sales.limit(10))

Original records: 1000008
Invalid records: 7
Cleaned records: 1000001
Cleaned dataset:


sales_id,customer_id,category,region,sales_amount,status
1,2,Clothing,Hyderabad,101.0,Completed
2,3,Furniture,Chennai,102.0,Completed
3,4,Groceries,Mumbai,103.0,Completed
4,5,Sports,Bengaluru,104.0,Completed
5,6,Electronics,Hyderabad,105.0,Completed
6,7,Clothing,Chennai,106.0,Completed
7,8,Furniture,Mumbai,107.0,Completed
8,9,Groceries,Bengaluru,108.0,Completed
9,10,Sports,Hyderabad,109.0,Completed
10,11,Electronics,Chennai,110.0,Cancelled


Run Final Data Quality Tests Using Assertions

In [0]:
# Final data quality validation using assert statements

assert cleaned_sales.filter(
    F.col("sales_id").isNull()
).count() == 0, "Missing sales IDs found!"

assert cleaned_sales.filter(
    F.col("customer_id").isNull() | (F.col("customer_id") <= 0)
).count() == 0, "Invalid customer IDs found!"

assert cleaned_sales.filter(
    F.col("sales_amount").isNull() | (F.col("sales_amount") <= 0)
).count() == 0, "Invalid sales amounts found!"

assert cleaned_sales.filter(
    ~F.col("category").isin(valid_categories)
).count() == 0, "Invalid categories found!"

assert cleaned_sales.filter(
    ~F.col("region").isin(valid_regions)
).count() == 0, "Invalid regions found!"

assert cleaned_sales.filter(
    ~F.col("status").isin(valid_statuses)
).count() == 0, "Invalid statuses found!"

assert duplicate_ids.count() == 1, "Unexpected duplicate test result!"

print("All final data quality assertions passed!")
print("Cleaned dataset is valid.")

All final data quality assertions passed!
Cleaned dataset is valid.


Generate the Final Data Quality Summary

In [0]:
# Final Data Quality Framework Summary

original_count = quality_df.count()
invalid_count = invalid_records.count()
cleaned_count = cleaned_sales.count()

print("=" * 50)
print("FINAL DATA QUALITY FRAMEWORK REPORT")
print("=" * 50)

print("Original Records       :", original_count)
print("Invalid Records        :", invalid_count)
print("Cleaned Records        :", cleaned_count)

print("-" * 50)

print("Missing Customer IDs   :", invalid_customers.count())
print("Duplicate Sales IDs    :", duplicate_ids.count())
print("Invalid Sales Amounts  :", invalid_amounts.count())
print("Invalid Categories     :", invalid_categories.count())
print("Invalid Regions        :", invalid_regions.count())
print("Invalid Statuses       :", invalid_statuses.count())

print("-" * 50)

print("Final Validation       : PASSED")
print("Data Quality Status    : SUCCESSFUL")
print("=" * 50)

FINAL DATA QUALITY FRAMEWORK REPORT
Original Records       : 1000008
Invalid Records        : 7
Cleaned Records        : 1000001
--------------------------------------------------
Missing Customer IDs   : 1
Duplicate Sales IDs    : 1
Invalid Sales Amounts  : 2
Invalid Categories     : 1
Invalid Regions        : 1
Invalid Statuses       : 1
--------------------------------------------------
Final Validation       : PASSED
Data Quality Status    : SUCCESSFUL


Create Reusable Validation Examples

In [0]:
from pyspark.sql import functions as F

# Reusable data validation function
def validate_data(df, rule_name, invalid_condition):
    failed_count = df.filter(invalid_condition).count()

    result = "PASS" if failed_count == 0 else "FAIL"

    print("Validation Rule:", rule_name)
    print("Failed Records:", failed_count)
    print("Result:", result)
    print("-" * 40)

    return failed_count


# Example 1: Null validation
null_check = validate_data(
    quality_df,
    "Customer ID Null Check",
    F.col("customer_id").isNull()
)

# Example 2: Range validation
range_check = validate_data(
    quality_df,
    "Sales Amount Range Check",
    F.col("sales_amount").isNull() |
    (F.col("sales_amount") <= 0)
)

# Example 3: Allowed values validation
category_check = validate_data(
    quality_df,
    "Category Validation",
    ~F.col("category").isin(valid_categories)
)

# Example 4: Duplicate validation
duplicate_check = duplicate_ids.count()

print("Duplicate Sales ID Check")
print("Duplicate IDs:", duplicate_check)
print("Result:", "PASS" if duplicate_check == 0 else "FAIL")

Validation Rule: Customer ID Null Check
Failed Records: 1
Result: FAIL
----------------------------------------
Validation Rule: Sales Amount Range Check
Failed Records: 2
Result: FAIL
----------------------------------------
Validation Rule: Category Validation
Failed Records: 1
Result: FAIL
----------------------------------------
Duplicate Sales ID Check
Duplicate IDs: 1
Result: FAIL
